# Gyara: fine-tune Hausa-ASR (free T4)

**Pick the data with `DATA` in the first code cell:**

- `DATA = "fleurs"` (default, no export needed): train on the whole FLEURS
  Hausa **train** split, pick the checkpoint on FLEURS **validation**, and
  measure base vs tuned **once** on FLEURS **test**. This shows the kit works
  end to end. It does not show what human corrections add: FLEURS is read
  speech, not corrected Gyara audio.
- `DATA = "export"`: train on a dataset exported from the Gyara correction app
  (`gyara export dataset`, zipped), with 30% FLEURS train mixed in.

What happens either way:

1. Training refuses to start if any clip or **speaker** of the training or dev
   data also appears in a held-out manifest (FLEURS test, plus your own
   held-out set if you have one). Do not weaken that check to make a run go.
   FLEURS publishes no speaker ids, so for FLEURS the check proves "no shared
   audio" and relies on the FLEURS card for speakers; `before_after.md` and the
   model card say so.
2. Defaults: LR 1e-5, 10% warmup, fp16, gradient checkpointing, light
   SpecAugment, early stopping on dev WER (checkpoints are picked on dev, never
   on test). The encoder is frozen only under 5 h of audio, so it trains in
   `fleurs` mode (about 12 h of audio).
3. Base and tuned models are evaluated **once** on held-out data and compared
   with a paired bootstrap. The result is printed as found, gain or not.

Bump `attempt` in the config every time you change a setting and re-train:
the count goes in the model card and `before_after.md`.

**Time on a free Colab T4, `DATA = "fleurs"` (estimated, not yet measured):**
install about 3 min; FLEURS download (test 0.8 GB, validation 0.35 GB, train
3.1 GB) 10 to 20 min; 600 training steps of 16 clips at an estimated 3 to 6 s
per step, 30 to 60 min; dev evaluation on 296 clips every 100 steps, 15 to 30
min in total; base and tuned on the 621 test clips about 5 min each (the
baseline took 304 s on a T4). **About 1.2 to 2 h in all, inside a 3 h budget.**
Keep the tab open: free Colab disconnects idle sessions. After about 50 steps
the progress bar shows the real speed; if it projects well over 3 h, stop and
see the note in section 3.

The last cell zips the results (never audio or weights) and downloads them.

*N-ATLAS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 0. Setup

In [ ]:
# ---- The only settings you may need to change ---------------------------------
DATA = "fleurs"      # "fleurs": FLEURS train only, no export needed
                     # "export": a zipped Gyara export (+30% FLEURS train)
SMOKE = False        # True: 2-step pipeline check with openai/whisper-tiny on a few
                     # clips (CPU is fine). Its numbers mean nothing; it only proves
                     # the cells run. Results go to runs/smoke-*, never committed.
GYARA_OWNER = "Baffaabba"          # GitHub user that hosts the repo
GYARA_BRANCH = "feat/foundation"   # change to "main" once merged
GYARA_EXTRAS = "train"
# --------------------------------------------------------------------------------
assert DATA in ("fleurs", "export"), DATA
GYARA_GIT = f"git+https://github.com/{GYARA_OWNER}/gyara@{GYARA_BRANCH}"
# FLEURS runs live under runs/*fleurs*/ so their predictions can be committed
# (docs/DECISIONS.md, 9 Oct). Own-audio runs must never be placed there.
OUT_ROOT = ("runs/smoke-finetune-" + DATA) if SMOKE else \
           ("runs/finetune-fleurs" if DATA == "fleurs" else "runs/finetune")
MODEL_NAME = "hausa-asr-gyara-fleurs" if DATA == "fleurs" else "hausa-asr-gyara"
OUT = f"{OUT_ROOT}/{MODEL_NAME}"

import importlib, importlib.util, os, pathlib, subprocess, sys

# Everything (data/, runs/) is written under WORK, so paths match the repo layout.
for base in ("/content", "/kaggle/working"):
    if os.path.isdir(base):
        WORK = pathlib.Path(base) / "gyara-work"
        break
else:
    WORK = pathlib.Path.cwd() / "gyara-work"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def pip_install(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-3000:])
    return r.returncode == 0

INSTALLED_FROM = None
if importlib.util.find_spec("gyara"):
    INSTALLED_FROM = "already installed in this session"
elif pip_install(f"gyara[{GYARA_EXTRAS}] @ {GYARA_GIT}"):
    INSTALLED_FROM = GYARA_GIT
else:
    print("\n>>> GitHub install FAILED (repo not pushed yet, wrong branch, or private). "
          "Run the next cell to install from a zip instead.")
print("working dir:", WORK)
print("gyara installed from:", INSTALLED_FROM)
print(f"DATA = {DATA!r}, SMOKE = {SMOKE}, results in {OUT}")

In [ ]:
# Fallback, ONLY if the cell above failed: install from a zip of the repo.
# On the laptop, in PowerShell, from E:\Repos\gyara:
#   Compress-Archive -Path gyara, pyproject.toml, README.md -DestinationPath gyara-src.zip -Force
# Then run this cell and pick gyara-src.zip. (Kaggle: add the zip as a dataset and set ZIP_PATH.)
ZIP_PATH = None
if INSTALLED_FROM is None:
    import zipfile
    if ZIP_PATH is None:
        from google.colab import files
        ZIP_PATH = next(iter(files.upload()))
    SRC = pathlib.Path("/content/gyara-src" if os.path.isdir("/content") else WORK.parent / "gyara-src")
    zipfile.ZipFile(ZIP_PATH).extractall(SRC)
    roots = sorted(p.parent for p in SRC.rglob("pyproject.toml"))
    assert roots, "no pyproject.toml in the zip: zip the repo root (gyara/, pyproject.toml, README.md)"
    assert pip_install("-e", f"{roots[0]}[{GYARA_EXTRAS}]"), "zip install failed; see pip output"
    sys.path.insert(0, str(roots[0]))  # editable installs are only importable after a restart
    importlib.invalidate_caches()
    INSTALLED_FROM = f"zip upload: {ZIP_PATH}"
print("gyara installed from:", INSTALLED_FROM)
!gyara --version

In [ ]:
# NCAIR1/* models are gated: accept the terms on each model page on huggingface.co
# (NCAIR1/Hausa-ASR) with the account that owns the token, then store a read token
# as a secret named HF_TOKEN
# (Colab: key icon in the left bar, and switch on "Notebook access"; Kaggle: Add-ons > Secrets).
import json, os, platform, zipfile, glob
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
if token:
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import login
    login(token=token)
else:
    assert SMOKE, "No HF_TOKEN secret found (NCAIR1/Hausa-ASR is gated)"
    print("No HF_TOKEN: fine for SMOKE (openai/whisper-tiny is public).")

import torch, transformers
import gyara
CUDA = torch.cuda.is_available()
assert CUDA or SMOKE, "No GPU: Runtime > Change runtime type > T4 GPU"
os.makedirs(f"{OUT_ROOT}/env", exist_ok=True)
try:
    from importlib.metadata import distribution
    direct_url = json.loads(distribution("gyara").read_text("direct_url.json") or "{}")
except Exception:
    direct_url = {}
ENV = {
    "data_mode": DATA, "smoke": SMOKE,
    "gyara_version": gyara.__version__,
    "installed_from": INSTALLED_FROM,
    "git_commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "gpu": torch.cuda.get_device_name(0) if CUDA else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if CUDA else None,
    "torch": torch.__version__, "transformers": transformers.__version__,
    "python": platform.python_version(),
}
json.dump(ENV, open(f"{OUT_ROOT}/env/env.json", "w"), indent=2)
print(json.dumps(ENV, indent=2))
if CUDA:
    !nvidia-smi | tee {OUT_ROOT}/env/nvidia-smi.txt

# Small text results only: never audio, never model weights or checkpoints.
KEEP = {".json", ".jsonl", ".md", ".srt", ".vtt", ".txt", ".yaml"}
def save_results(zip_name="gyara-results.zip", download=True):
    """Zip the small result files under OUT_ROOT (paths kept: unzip at the repo root)."""
    files = [p for p in pathlib.Path(OUT_ROOT).rglob("*")
             if p.is_file() and p.suffix in KEEP
             and "checkpoints" not in p.parts
             and ("model" not in p.parts or p.name == "README.md")]  # model card only
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.as_posix())
    print(f"{zip_name}: {len(files)} files")
    for p in sorted(files):
        print("  ", p.as_posix())
    if download:
        try:
            from google.colab import files as colab_files
            colab_files.download(zip_name)
        except Exception:
            print(f"Not on Colab: download {WORK / zip_name} from the file browser / Output tab.")

## 1. FLEURS Hausa (ha_ng): held-out test, dev, training

Test is the held-out set in both modes. `fleurs` mode also needs train (to
learn from) and validation (to pick the checkpoint). A split already on disk is
not downloaded again, so this cell is safe to re-run after a disconnect.

In [ ]:
TEST, FLEURS_TRAIN, FLEURS_DEV = ("data/fleurs/test.jsonl", "data/fleurs/train.jsonl",
                                  "data/fleurs/validation.jsonl")
LIM = {"test": 16, "train": 24, "validation": 8} if SMOKE else {}
for split, path in (("test", TEST), ("train", FLEURS_TRAIN), ("validation", FLEURS_DEV)):
    if split == "validation" and DATA != "fleurs":
        continue  # export mode splits dev off the export by speaker
    if os.path.exists(path):
        print(f"{path}: already here, not fetched again")
        continue
    limit = f"--limit {LIM[split]}" if split in LIM else ""
    !gyara fetch-fleurs --split {split} {limit}
    assert os.path.exists(path), f"fetching FLEURS {split} failed; see the output above"
for s in ("test", "train", "validation"):
    p = f"data/fleurs/{s}.summary.json"
    if os.path.exists(p):
        print(s, open(p, encoding="utf-8").read())

## 2. Upload your Gyara export (`DATA = "export"` only)

In [ ]:
EXPORT_DIR = None
EXPORT_ZIP = None  # Kaggle: set to the path of the zip under /kaggle/input/...
if DATA != "export":
    print(f"DATA = {DATA!r}: no export needed, nothing to upload.")
else:
    if EXPORT_ZIP is None:
        from google.colab import files
        EXPORT_ZIP = next(iter(files.upload()))
    EXPORT_DIR = pathlib.Path("exports") / pathlib.Path(EXPORT_ZIP).stem
    zipfile.ZipFile(EXPORT_ZIP).extractall(EXPORT_DIR)
    # The export may be zipped with or without its top-level folder. The trainer reads
    # manifest.jsonl, or the HF metadata.jsonl if that is all there is.
    hits = list(EXPORT_DIR.rglob("manifest.jsonl")) or list(EXPORT_DIR.rglob("metadata.jsonl"))
    assert hits, "no manifest.jsonl or metadata.jsonl in the zip: is this a gyara export?"
    EXPORT_DIR = hits[0].parent
    print(EXPORT_DIR, sum(1 for _ in open(hits[0], encoding="utf-8")), "rows in", hits[0].name)

## 3. Config

`LANGUAGE` matches the benchmark run: `chosen_language` in
`runs/g0-smoke/decoding_choice.json` is `"hausa"` (chosen on FLEURS validation,
test never used). `"auto"` there would mean `None` here.

`fleurs` mode, attempt 1: all 3,259 FLEURS train clips (`fleurs_fraction: 1.0`),
dev = FLEURS validation (296 clips; its speakers differ from train according to
the FLEURS card, there are no speaker ids to check), 600 steps of 16 clips
(about 3 epochs), dev WER every 100 steps, early stopping after 3 evaluations
without improvement. These were set before any run, not tuned.

If the progress bar projects far over the budget, stop the cell, lower
`MAX_STEPS`, and re-run this cell and the next. That is still attempt 1 as long
as no dev or test number was printed before you stopped; write the change in
`NOTES`. Any change after a number was seen is attempt 2.

In [ ]:
import yaml
LANGUAGE = "hausa"   # runs/g0-smoke/decoding_choice.json: chosen_language
OWN_HELDOUT = None   # optional: your own frozen held-out manifest (>= 30 min, other speakers)
ATTEMPT = 1          # bump on every re-train with changed settings
MAX_STEPS = 600 if DATA == "fleurs" else 1000
NOTES = ""           # anything changed by hand, and why
if DATA == "fleurs":
    data_cfg = {
        "exports": [],
        "fleurs_fraction": 1.0,        # FLEURS train only: every train clip is used
        "fleurs_manifest": FLEURS_TRAIN,
        "dev_manifest": FLEURS_DEV,    # checkpoint choice; never the test set
    }
else:
    data_cfg = {
        "exports": [str(EXPORT_DIR)],
        "fleurs_fraction": 0.3,        # 30% FLEURS train to limit forgetting
        "fleurs_manifest": FLEURS_TRAIN,
        "dev_fraction": 0.1,           # speaker-disjoint split of the export
    }
data_cfg["heldout"] = [TEST] + ([OWN_HELDOUT] if OWN_HELDOUT else [])
config = {
    "attempt": ATTEMPT,
    "base_model": "NCAIR1/Hausa-ASR",
    "language": LANGUAGE,
    "output_dir": OUT,
    "model_name": MODEL_NAME,
    "data": data_cfg,
    "training": {"max_steps": MAX_STEPS, "eval_steps": 100, "early_stopping_patience": 3,
                 "per_device_train_batch_size": 8, "per_device_eval_batch_size": 8,
                 "gradient_accumulation_steps": 2, "gradient_checkpointing": True},
    "eval": {"after": True, "batch_size": 8},
    "notes": NOTES,
}
if SMOKE:  # pipeline check only: tiny public model, 2 steps, 8 test clips
    config.update(base_model="openai/whisper-tiny", processor_fallback="openai/whisper-tiny")
    config["training"].update(max_steps=2, eval_steps=1, per_device_train_batch_size=2,
                              per_device_eval_batch_size=2, gradient_accumulation_steps=1,
                              generation_max_length=32, logging_steps=1)
    config["eval"].update(limit=8, batch_size=4)
os.makedirs(OUT, exist_ok=True)
yaml.safe_dump(config, open("finetune.yaml", "w", encoding="utf-8"), sort_keys=False, allow_unicode=True)
yaml.safe_dump(config, open(f"{OUT}/config.yaml", "w", encoding="utf-8"), sort_keys=False, allow_unicode=True)
print(open("finetune.yaml", encoding="utf-8").read())

## 4. Train (leakage check first)

In [ ]:
!gyara finetune --config finetune.yaml
assert os.path.exists(f"{OUT}/summary.json"), "training did not finish; see the output above"
save_results("gyara-finetune-results.zip", download=False)  # survives a disconnect of the next cells

## 5. Before / after on held-out data

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/before_after.md", encoding="utf-8").read()))

In [ ]:
# The same paired comparison through the CLI, for the FLEURS test set.
# Writes compare.md + compare.json into the tuned run's folder.
tuned = sorted(p.as_posix() for p in pathlib.Path(OUT).glob(f"eval/{MODEL_NAME}-a*-test"))
base = sorted(p.as_posix() for p in pathlib.Path(OUT).glob("eval/base-test"))
print(tuned, base)
if tuned and base:
    !gyara compare {tuned[-1]} {base[0]} > /dev/null
    display(Markdown(open(f"{tuned[-1]}/compare.md", encoding="utf-8").read()))

## 6. (Optional) publish

Not part of the 11 Oct submission (publishing the model is cut). The model
keeps the N-ATLAS licence of its base (free use capped at 1,000 active users; a
renamed derivative must say "Powered by Awarri"). The model card in
`<OUT>/model/README.md` already says so.

In [ ]:
# from huggingface_hub import HfApi
# HfApi().upload_folder(folder_path=f"{OUT}/model",
#                       repo_id="<your-user>/hausa-asr-gyara-powered-by-awarri", private=True)
print(open(f"{OUT}/model/README.md", encoding="utf-8").read())

## 7. Save the results

Zips the text results under `OUT_ROOT` (no audio, no weights, no checkpoints)
and downloads them. Unzip at the repo root. In `fleurs` mode you get
`runs/finetune-fleurs/hausa-asr-gyara-fleurs/`: `before_after.md`,
`summary.json`, `config.yaml`, `model/README.md` (the model card), and
`eval/base-test/` + `eval/hausa-asr-gyara-fleurs-a1-test/` with `metrics.json`,
`report.md`, `meta.json`, `predictions.jsonl` (FLEURS predictions are committed),
plus `compare.json`/`compare.md`; and `runs/finetune-fleurs/env/` (GPU,
versions, install source). The weights stay in `<OUT>/model/`: copy them to
Drive before the session ends if you want to keep them.

In [ ]:
save_results("gyara-finetune-results.zip")